# Dictionary Extensions

Runnable samples for every method in `CSharpHelperExtensions.Dictionaries`.  
Run the **Setup** cell first, then any section independently.

| Section | Methods |
|---|---|
| [1. Safe Value Lookup](#1-safe-value-lookup) | `GetValueOrDefault` |
| [2. Add-If-Missing](#2-add-if-missing) | `GetOrAdd` |
| [3. Merging Dictionaries](#3-merging-dictionaries) | `Merge` |
| [4. Bulk Add](#4-bulk-add) | `AddRange` |
| [5. Filtering In-Place](#5-filtering-in-place) | `RemoveWhere` |
| [6. Read-Only View](#6-read-only-view) | `AsReadOnly` |
| [7. Chaining Examples](#7-chaining-examples) | Composing multiple extensions into pipelines |

## Setup

> **Run this cell first.** It loads the compiled library and imports the required namespaces.
>
> Build first if the DLL is missing: `dotnet build` from the repo root.

In [2]:
#r "../src/CSharpHelperExtensions/bin/Debug/net10.0/CSharpHelperExtensions.dll"
using System.Collections.Generic;
using CSharpHelperExtensions.Dictionaries;   // all DictionaryExtensions

---
## 1. Safe Value Lookup

| Method | Signature | Returns |
|---|---|---|
| `GetValueOrDefault` | `IDictionary<TKey,TValue>?, TKey → TValue?` | value for key, or `default` when dict/key is null or key not found |

> **Note:** `GetValueOrDefault` is called using static syntax (`DictionaryExtensions.GetValueOrDefault(dict, key)`) rather than extension syntax (`dict.GetValueOrDefault(key)`). This is required because the BCL ships `CollectionExtensions.GetValueOrDefault` on `IReadOnlyDictionary<TKey,TValue>` — since `Dictionary<TKey,TValue>` implements both interfaces, the extension call is ambiguous and won't compile.

In [3]:
// GetValueOrDefault — returns value when key exists
var scores = new Dictionary<string, int>
{
    ["alice"] = 95,
    ["bob"]   = 82,
};

display(DictionaryExtensions.GetValueOrDefault(scores, "alice"));   // 95
display(DictionaryExtensions.GetValueOrDefault(scores, "bob"));     // 82

95

82

In [4]:
// GetValueOrDefault — returns default(TValue) when key is not found
display(DictionaryExtensions.GetValueOrDefault(scores, "charlie"));  // 0  (default for int)

var labels = new Dictionary<string, string> { ["en"] = "Hello" };
display(DictionaryExtensions.GetValueOrDefault(labels, "fr"));       // null  (default for string)

0

<null>

In [5]:
// GetValueOrDefault — null-safe: returns default when dict is null
IDictionary<string, int>? nullDict = null;
display(DictionaryExtensions.GetValueOrDefault(nullDict, "any"));    // 0  (no NullReferenceException)

// Also null-safe on key
display(DictionaryExtensions.GetValueOrDefault(scores, null!));       // 0  (null key → default)

0

0


(2,25): warning CS8632: The annotation for nullable reference types should only be used in code within a '#nullable' annotations context.



---
## 2. Add-If-Missing

| Method | Signature | Returns |
|---|---|---|
| `GetOrAdd` | `IDictionary<TKey,TValue>, TKey, Func<TKey,TValue> → TValue` | existing value if key present; otherwise invokes factory, stores result, and returns it |

In [6]:
// GetOrAdd — returns existing value; factory is NOT called when key already exists
var cache = new Dictionary<string, string> { ["user:1"] = "Alice" };

var factoryCalled = false;
var result = cache.GetOrAdd("user:1", key =>
{
    factoryCalled = true;
    return $"fetched-{key}";
});

display(result);         // "Alice"   (existing value returned)
display(factoryCalled);  // False     (factory skipped)

Alice

False

In [7]:
// GetOrAdd — key missing: factory is called, result stored and returned
var result2 = cache.GetOrAdd("user:2", key => $"fetched-{key}");

display(result2);           // "fetched-user:2"
display(cache["user:2"]);   // "fetched-user:2"  (stored in dictionary)

fetched-user:2

fetched-user:2

In [8]:
// GetOrAdd — practical use: lazy-initialise nested collections
var grouped = new Dictionary<string, List<int>>();

void Add(string group, int value)
    => grouped.GetOrAdd(group, _ => new List<int>()).Add(value);

Add("evens", 2); Add("odds", 1);
Add("evens", 4); Add("odds", 3);
Add("evens", 6);

display(grouped["evens"]);   // [2, 4, 6]
display(grouped["odds"]);    // [1, 3]

[ 2, 4, 6 ]

[ 1, 3 ]

In [9]:
// GetOrAdd — throws ArgumentNullException for null key or null factory
var d = new Dictionary<string, int>();

try { d.GetOrAdd(null!, _ => 1); }
catch (ArgumentNullException e) { display($"null key: {e.ParamName}"); }      // null key: key

try { d.GetOrAdd("k", null!); }
catch (ArgumentNullException e) { display($"null factory: {e.ParamName}"); }  // null factory: factory

null key: key

null factory: factory

---
## 3. Merging Dictionaries

| Method | Signature | Returns |
|---|---|---|
| `Merge` | `IDictionary<TKey,TValue>, IDictionary<TKey,TValue>?, bool overwrite=false → IDictionary<TKey,TValue>` | original dict (mutated in-place) with all entries from `other` added |

Returns the original dictionary for fluent chaining. A null `other` is silently ignored.

In [10]:
// Merge — no overlapping keys: all entries from other are added
var defaults = new Dictionary<string, int> { ["timeout"] = 30, ["retries"] = 3 };
var overrides = new Dictionary<string, int> { ["pageSize"] = 50 };

defaults.Merge(overrides);
display(defaults);   // { timeout: 30, retries: 3, pageSize: 50 }

key,value
timeout,30
retries,3
pageSize,50


In [11]:
// Merge — overwrite: false (default) skips duplicate keys; existing values are kept
var config = new Dictionary<string, string> { ["theme"] = "light", ["lang"] = "en" };
var userPrefs = new Dictionary<string, string> { ["theme"] = "dark", ["fontSize"] = "14" };

config.Merge(userPrefs, overwrite: false);
display(config["theme"]);     // "light"   (existing value kept)
display(config["fontSize"]);  // "14"      (new key added)

light

14

In [12]:
// Merge — overwrite: true replaces existing values with values from other
var base1 = new Dictionary<string, string> { ["theme"] = "light", ["lang"] = "en" };
var user1  = new Dictionary<string, string> { ["theme"] = "dark",  ["fontSize"] = "14" };

base1.Merge(user1, overwrite: true);
display(base1["theme"]);     // "dark"   (overwritten)
display(base1["fontSize"]);  // "14"     (new key added)

dark

14

In [13]:
// Merge — null other is silently ignored; returns same dict instance for chaining
var dict1 = new Dictionary<string, int> { ["a"] = 1 };
var result3 = dict1.Merge(null!);

display(object.ReferenceEquals(dict1, result3));  // True   (same instance)
display(dict1.Count);                             // 1      (unchanged)

True

1

---
## 4. Bulk Add

| Method | Signature | Returns |
|---|---|---|
| `AddRange` | `IDictionary<TKey,TValue>, IEnumerable<KeyValuePair<TKey,TValue>>?, bool overwrite=false → IDictionary<TKey,TValue>` | original dict (mutated in-place) with all pairs added |

Works with any `IEnumerable<KeyValuePair<TKey,TValue>>` — including another dictionary, a list of tuples converted to KVPs, or LINQ projections. A null source is silently ignored.

In [14]:
// AddRange — adds all pairs from a list; returns same dict for chaining
var inventory = new Dictionary<string, int> { ["apple"] = 10 };
var incoming = new List<KeyValuePair<string, int>>
{
    new("banana", 5),
    new("cherry", 20),
};

inventory.AddRange(incoming);
display(inventory);   // { apple: 10, banana: 5, cherry: 20 }

key,value
apple,10
banana,5
cherry,20


In [15]:
// AddRange — overwrite: false (default) keeps existing values for duplicate keys
var inv2 = new Dictionary<string, int> { ["apple"] = 10, ["banana"] = 5 };
var restock = new List<KeyValuePair<string, int>> { new("apple", 99), new("mango", 15) };

inv2.AddRange(restock, overwrite: false);
display(inv2["apple"]);   // 10   (existing kept)
display(inv2["mango"]);   // 15   (new key added)

10

15

In [16]:
// AddRange — overwrite: true replaces existing values
var inv3 = new Dictionary<string, int> { ["apple"] = 10, ["banana"] = 5 };
var restock2 = new List<KeyValuePair<string, int>> { new("apple", 99), new("mango", 15) };

inv3.AddRange(restock2, overwrite: true);
display(inv3["apple"]);   // 99   (overwritten)
display(inv3["mango"]);   // 15

99

15

In [17]:
// AddRange — works with any IEnumerable<KeyValuePair>; here from a LINQ projection
var codes = new[] { "USD", "GBP", "EUR" };
var rates = new Dictionary<string, double>();

rates.AddRange(codes.Select((c, i) => new KeyValuePair<string, double>(c, 1.0 + i * 0.1)));
display(rates);   // { USD: 1.0, GBP: 1.1, EUR: 1.2 }

key,value
USD,1
GBP,1.1
EUR,1.2


In [18]:
// AddRange — null pairs are silently ignored
var inv4 = new Dictionary<string, int> { ["a"] = 1 };
inv4.AddRange(null!);
display(inv4.Count);   // 1  (unchanged, no exception)

1

---
## 5. Filtering In-Place

| Method | Signature | Returns |
|---|---|---|
| `RemoveWhere` | `IDictionary<TKey,TValue>, Func<TKey,bool> → IDictionary<TKey,TValue>` | original dict (mutated in-place) with matching keys removed |

Collects matching keys first, then removes them — safe against the "collection modified during enumeration" error for the keys being removed. The predicate receives each key.

In [19]:
// RemoveWhere — removes all keys matching the predicate
var settings = new Dictionary<string, string>
{
    ["db.host"]    = "localhost",
    ["db.port"]    = "5432",
    ["cache.host"] = "redis",
    ["cache.ttl"]  = "300",
    ["app.name"]   = "MyApp",
};

settings.RemoveWhere(k => k.StartsWith("cache."));
display(settings);   // { db.host: localhost, db.port: 5432, app.name: MyApp }

key,value
db.host,localhost
db.port,5432
app.name,MyApp


In [20]:
// RemoveWhere — no matching keys: dict is unchanged
var nums = new Dictionary<string, int> { ["a"] = 1, ["b"] = 2 };
nums.RemoveWhere(k => k == "z");
display(nums.Count);   // 2  (unchanged)

2

In [21]:
// RemoveWhere — returns same dict instance for chaining
var env = new Dictionary<string, string>
{
    ["DEBUG"]   = "true",
    ["SECRET"]  = "s3cr3t",
    ["HOST"]    = "prod.example.com",
    ["API_KEY"] = "abc123",
};

// Remove sensitive keys, then check what's left
var safe = env.RemoveWhere(k => k.Contains("KEY") || k.Contains("SECRET"));
display(object.ReferenceEquals(env, safe));   // True   (same instance)
display(safe);                                 // { DEBUG: true, HOST: prod.example.com }

True

key,value
DEBUG,true
HOST,prod.example.com


In [22]:
// RemoveWhere — throws ArgumentNullException for null predicate
var d2 = new Dictionary<string, int> { ["a"] = 1 };

try { d2.RemoveWhere(null!); }
catch (ArgumentNullException e) { display($"null predicate: {e.ParamName}"); }  // predicate

null predicate: predicate

---
## 6. Read-Only View

| Method | Signature | Returns |
|---|---|---|
| `AsReadOnly` | `IDictionary<TKey,TValue> where TKey:notnull → IReadOnlyDictionary<TKey,TValue>` | a live read-only view over the original dictionary |

> **Live view, not a copy.** Mutations to the underlying dictionary are visible through the returned `IReadOnlyDictionary`. Pass it to subsystems that should read but not write the data.
>
> **Note:** Called using static syntax (`DictionaryExtensions.AsReadOnly(dict)`) to disambiguate from `System.Collections.Generic.CollectionExtensions.AsReadOnly`.
>
> **Constraint:** `TKey` must be non-nullable (`where TKey : notnull`) — a requirement of the underlying `ReadOnlyDictionary<TKey,TValue>`.

In [23]:
// AsReadOnly — wraps the dict as IReadOnlyDictionary; reads work normally
var prices = new Dictionary<string, decimal>
{
    ["apple"]  = 1.20m,
    ["banana"] = 0.50m,
};

IReadOnlyDictionary<string, decimal> readOnly = DictionaryExtensions.AsReadOnly(prices);
display(readOnly.Count);         // 2
display(readOnly["apple"]);      // 1.20
display(readOnly.ContainsKey("banana"));  // True

2

1.20

True

In [24]:
// AsReadOnly — live view: mutations to the underlying dict are visible through the wrapper
var source = new Dictionary<string, int> { ["a"] = 1 };
var view   = DictionaryExtensions.AsReadOnly(source);

display(view.Count);   // 1

source["b"] = 99;      // mutate the underlying dict
display(view.Count);   // 2   (change is visible through the view)
display(view["b"]);    // 99

1

2

99

In [25]:
// AsReadOnly — throws ArgumentNullException for null dict
try { DictionaryExtensions.AsReadOnly<string, int>(null!); }
catch (ArgumentNullException e) { display($"null dict: {e.ParamName}"); }  // dict

null dict: dict

---
## 7. Chaining Examples

`Merge`, `AddRange`, and `RemoveWhere` all return the same dictionary instance, making them chainable. The examples below show realistic pipelines that compose several methods.

### Build a config from defaults + environment overrides, then expose read-only
`Merge → RemoveWhere → AsReadOnly`

In [26]:
var appDefaults = new Dictionary<string, string>
{
    ["log.level"]  = "info",
    ["log.format"] = "json",
    ["db.pool"]    = "5",
    ["debug"]      = "false",
};

var envVars = new Dictionary<string, string>
{
    ["log.level"] = "warn",   // override log level
    ["db.pool"]   = "20",     // override pool size
    ["SECRET_KEY"] = "abc",   // sensitive — should be stripped
};

// Merge env overrides, strip sensitive keys, expose as read-only
var config = DictionaryExtensions.AsReadOnly(
    appDefaults
        .Merge(envVars, overwrite: true)
        .RemoveWhere(k => k.Contains("SECRET"))
);

display(config["log.level"]);   // "warn"   (overridden by env)
display(config["db.pool"]);     // "20"     (overridden by env)
display(config["log.format"]);  // "json"   (default kept)
display(config.ContainsKey("SECRET_KEY"));  // False  (stripped)

warn

20

json

False

### Bulk-populate a registry, then filter obsolete entries
`AddRange → RemoveWhere`

In [27]:
// Start with some registered handlers
var handlers = new Dictionary<string, string> { ["v1/orders"] = "OrderHandlerV1" };

// Add new routes in bulk
var newRoutes = new List<KeyValuePair<string, string>>
{
    new("v2/orders",   "OrderHandlerV2"),
    new("v2/products", "ProductHandlerV2"),
    new("v1/products", "ProductHandlerV1"),   // legacy — will be pruned
    new("health",      "HealthHandler"),
};

// Add routes then immediately prune the old v1 ones
handlers
    .AddRange(newRoutes)
    .RemoveWhere(k => k.StartsWith("v1/"));

display(handlers);
// { v2/orders: OrderHandlerV2, v2/products: ProductHandlerV2, health: HealthHandler }

key,value
v2/orders,OrderHandlerV2
v2/products,ProductHandlerV2
health,HealthHandler


### Lazy-initialise grouped buckets with GetOrAdd, then read-only snapshot
`GetOrAdd → AddRange → AsReadOnly`

In [28]:
// Accumulate events into per-type buckets
var buckets = new Dictionary<string, List<string>>();

var events = new[]
{
    ("order",   "order-1001"),
    ("payment", "pay-501"),
    ("order",   "order-1002"),
    ("payment", "pay-502"),
    ("shipment","ship-201"),
};

foreach (var (type, id) in events)
    buckets.GetOrAdd(type, _ => new List<string>()).Add(id);

// Expose as read-only before passing to another layer
var snapshot = DictionaryExtensions.AsReadOnly(buckets);

display(snapshot["order"]);    // ["order-1001", "order-1002"]
display(snapshot["payment"]);  // ["pay-501", "pay-502"]
display(snapshot["shipment"]); // ["ship-201"]

[ order-1001, order-1002 ]

[ pay-501, pay-502 ]

[ ship-201 ]

### Merge multiple sources in priority order
`Merge (overwrite:false) × 3`

In [29]:
// Highest-priority source first; each subsequent merge skips already-set keys
var commandLine = new Dictionary<string, string> { ["port"] = "9090" };
var envVars2   = new Dictionary<string, string> { ["port"] = "8080", ["host"] = "prod.example.com" };
var fileConfig = new Dictionary<string, string> { ["port"] = "3000", ["host"] = "localhost", ["debug"] = "false" };

// Start with CLI args (highest priority), layer env vars, then file config
var resolved = commandLine
    .Merge(envVars2,   overwrite: false)   // env can't override CLI
    .Merge(fileConfig, overwrite: false);  // file can't override CLI or env

display(resolved["port"]);    // "9090"             (CLI wins)
display(resolved["host"]);    // "prod.example.com" (env wins over file)
display(resolved["debug"]);   // "false"            (only in file)

9090

prod.example.com

false

### Safe lookup with fallback using GetOrAdd as a memoisation cache
`GetValueOrDefault → GetOrAdd`

In [30]:
// Cheap probe with GetValueOrDefault before the more expensive GetOrAdd
var memo = new Dictionary<int, long>();

long Fib(int n)
{
    if (n <= 1) return n;
    var cached = DictionaryExtensions.GetValueOrDefault(memo, n);
    if (cached != 0) return cached;                      // already computed
    return memo.GetOrAdd(n, k => Fib(k - 1) + Fib(k - 2));
}

display(Fib(10));   // 55
display(Fib(20));   // 6765
display(memo.Count); // number of values cached

55

6765

19